In [54]:
import nltk 
from nltk.tokenize import word_tokenize
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
import gensim.downloader as api
import random

In [55]:
df = pd.read_csv("hf://datasets/gsri-18/ISEAR-dataset-complete/ISEAR_dataset_complete.csv")
df = df.drop(columns=["Unnamed: 2"])
content = df["content"]
print(content.head())
emotions = df["emotion"].unique()

# Representation 1: Tokenization
tokens = []
for element in df["content"]:
    tokenized_sentence = word_tokenize(element)
    tokens.append(tokenized_sentence)
df["tokens"] = tokens


0    On days when I feel close to my partner and ot...
1    Every time I imagine that someone I love or I ...
2    When I had been obviously unjustly treated and...
3    When I think about the short time that we live...
4    At a gathering I found myself involuntarily si...
Name: content, dtype: str


In [56]:
# Embedding
from gensim.models import Word2Vec

model_CBOW = Word2Vec(
    sentences=df["tokens"],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=0,
    epochs=10
)

model_S = Word2Vec(
    sentences=df["tokens"],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10
)

In [57]:
# All unique words in your dataset
dataset_vocab = set(
    word
    for sentence in df["tokens"]
    for word in sentence
)

# Words represented in the Word2Vec model
model_vocab = set(model_CBOW.wv.key_to_index.keys())

# Words present in both vocabularies
covered_words = dataset_vocab.intersection(model_vocab)

# Words missing from the model
oov_words = dataset_vocab - model_vocab

# Calculate coverage
coverage = len(covered_words) / len(dataset_vocab) * 100

print(f"Dataset vocabulary: {len(dataset_vocab)}")
print(f"Model vocabulary: {len(model_vocab)}")
print(f"Covered words: {len(covered_words)}")
print(f"OOV words: {len(oov_words)}")
print(f"Vocabulary coverage: {coverage:.2f}%")

print("Missing words:", list(oov_words)[:30])

Dataset vocabulary: 10173
Model vocabulary: 4971
Covered words: 4971
OOV words: 5202
Vocabulary coverage: 48.86%
Missing words: ['ants', 'leaf', 'acceptted', 'dissertation', 'centuries', 'ingratiating', 'obstacles', 'suspicion', 'desagreable', 'pre-set', 'calamity', 'veterinarian', 'thourgh', 'ideal', 'Hurt', 'Chirimba', 'whash', 'Unfair', 'Mugged', 'trunk', 'M.Phil', '2/3', 'student-like', 'prosperity', 'persoon', 'Mathematics', 'temporary', 'al', 'Begining', 'striven']


In [58]:
# All unique words in your dataset
dataset_vocab = set(
    word
    for sentence in df["tokens"]
    for word in sentence
)

# Words represented in the Word2Vec model
model_vocab = set(model_S.wv.key_to_index.keys())

# Words present in both vocabularies
covered_words = dataset_vocab.intersection(model_vocab)

# Words missing from the model
oov_words = dataset_vocab - model_vocab

# Calculate coverage
coverage = len(covered_words) / len(dataset_vocab) * 100

print(f"Dataset vocabulary: {len(dataset_vocab)}")
print(f"Model vocabulary: {len(model_vocab)}")
print(f"Covered words: {len(covered_words)}")
print(f"OOV words: {len(oov_words)}")
print(f"Vocabulary coverage: {coverage:.2f}%")

print("Missing words:", list(oov_words)[:30])

Dataset vocabulary: 10173
Model vocabulary: 4971
Covered words: 4971
OOV words: 5202
Vocabulary coverage: 48.86%
Missing words: ['ants', 'leaf', 'acceptted', 'dissertation', 'centuries', 'ingratiating', 'obstacles', 'suspicion', 'desagreable', 'pre-set', 'calamity', 'veterinarian', 'thourgh', 'ideal', 'Hurt', 'Chirimba', 'whash', 'Unfair', 'Mugged', 'trunk', 'M.Phil', '2/3', 'student-like', 'prosperity', 'persoon', 'Mathematics', 'temporary', 'al', 'Begining', 'striven']


In [59]:
total_tokens = sum(
    len(sentence)
    for sentence in df["tokens"]
)

covered_tokens = sum(
    1
    for sentence in df["tokens"]
    for word in sentence
    if word in model_CBOW.wv
)

token_coverage = covered_tokens / total_tokens * 100

print(f"Token-level coverage CBOW: {token_coverage:.2f}%")

Token-level coverage CBOW: 97.18%


In [60]:
total_tokens = sum(
    len(sentence)
    for sentence in df["tokens"]
)

covered_tokens = sum(
    1
    for sentence in df["tokens"]
    for word in sentence
    if word in model_S.wv
)

token_coverage = covered_tokens / total_tokens * 100

print(f"Token-level coverage S: {token_coverage:.2f}%")

Token-level coverage S: 97.18%


In [61]:
print(model_CBOW.wv.most_similar("leukemia", topn=10))
print(model_S.wv.most_similar("leukemia", topn=10))

[('seriously', 0.9528579115867615), ('stealing', 0.9511541128158569), ('cerebral', 0.9498323798179626), ('robbed', 0.9497810006141663), ('beloved', 0.9488080143928528), ('severe', 0.9471285343170166), ('wristwatch', 0.9445464015007019), ('catched', 0.942639946937561), ('St.', 0.9407409429550171), ('empty', 0.940635621547699)]
[('gradmother', 0.9559885263442993), ('paternal', 0.9489077925682068), ('tragically', 0.9461374878883362), ('cerebral', 0.9450694918632507), ('thirteen', 0.9445726275444031), ('lung', 0.9443057775497437), ('collapsed', 0.9382310509681702), ('August', 0.934075117111206), ('horse', 0.9302135705947876), ('discharged', 0.9297960996627808)]


In [62]:
print([word for word in model_vocab if len(word) <= 2])
print([word for word in dataset_vocab if len(word) <= 2])

['km', 'TE', "'m", '80', 'my', 'on', '``', 'in', 'No', '!', '20', 'he', '%', 'As', 'am', 'He', '9', "'", '[', ']', '4', 'e', 'of', 'OK', 'ca', 'we', 'pm', 'X', ')', '8', 'Ca', 'is', 'it', '(', '-', '$', 'B+', '11', 'an', 'do', 'Of', '16', 'if', 'My', '1', 'a', 'Ye', ';', '2A', 'as', 'to', '10', '5', 'up', 'Mr', 'so', '22', 'It', 'By', '12', "''", '3', 'An', '13', 'C', '35', '19', ':', '14', '60', 'Is', '.', 'go', ',', '84', "'s", 'NO', 'ex', 'i', 'aa', '18', 'or', 'B', 'at', '7', 'be', '30', 'me', 'To', 'us', 'by', '17', 'TV', '40', '&', 'So', 'PY', '15', '21', 'Oh', 'I.', 'If', 'ok', 'F4', '?', 'Km', 'A', 'LP', '/', '50', 'We', 'wo', 'TA', 'no', 'At', 'II', '6', 'C+', 'I', 'On', 'm', '2', "'d", 'In', 'D']
['km', 'al', 't', 'TE', "'m", 'M.', 'ne', 'WD', '80', 'Mt', '53', 'Do', 'my', 'on', '5B', 'Rs', '``', '31', '+2', 'in', 'No', '!', '20', 'El', '79', 'F', 'he', '%', 'm.', 'As', 'T', 'am', 'LA', 'He', '9', '95', "'", '[', ']', '4', 'e', 'of', 'E', 'OK', 'ca', 'we', 'hi', 'Kg', 'pm', '

In [63]:
# Pooling using mean
def mean_pool(tokens, word_vectors):
    vectors = [
        word_vectors[token]
        for token in tokens
        if token in word_vectors
    ]

    if not vectors:
        return np.zeros(word_vectors.vector_size)

    return np.mean(vectors, axis=0)

# Pooling using max
def max_pool(tokens, word_vectors):
    vectors = [
        word_vectors[token]
        for token in tokens
        if token in word_vectors
    ]

    if not vectors:
        return np.zeros(word_vectors.vector_size)

    return np.max(vectors, axis=0)

# Create a mean and max pooled vector for each document, for both embedding models
X_mean_cbow = np.vstack([
    mean_pool(tokens, model_CBOW.wv)
    for tokens in df["tokens"]
])

X_mean_skipgram = np.vstack([
    mean_pool(tokens, model_S.wv)
    for tokens in df["tokens"]
])

X_max_cbow = np.vstack([
    max_pool(tokens, model_CBOW.wv)
    for tokens in df["tokens"]
])

X_max_skipgram = np.vstack([
    max_pool(tokens, model_S.wv)
    for tokens in df["tokens"]
])

# Vector shapes should be the same
print(X_mean_cbow.shape)
print(X_mean_skipgram.shape)
print(X_max_cbow.shape)
print(X_max_skipgram.shape)


(7516, 100)
(7516, 100)
(7516, 100)
(7516, 100)


In [64]:
# Training classifiers
y = df["emotion"]

implementations = {
    "CBOW + mean": X_mean_cbow,
    "Skip-gram + mean": X_mean_skipgram,
    "CBOW + max": X_max_cbow,
    "Skip-gram + max": X_max_skipgram,
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = {}

for name, X in implementations.items():
    classifier = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    scores = cross_val_score(
        classifier,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    results[name] = scores
    print(
        f"{name}: "
        f"mean={scores.mean():.4f}, "
        f"std={scores.std():.4f}"
    )

CBOW + mean: mean=0.3489, std=0.0153
Skip-gram + mean: mean=0.4324, std=0.0100
CBOW + max: mean=0.3278, std=0.0119
Skip-gram + max: mean=0.3695, std=0.0054


In [65]:
# Now trying the same but using Glove instead
glove = api.load("glove-wiki-gigaword-100")
vector_size = glove.vector_size

# Glove works in lowercase
df["glove_tokens"] = [
    [token.lower() for token in tokens]
    for tokens in df["tokens"]
]

# Pooling functions but with glove:
def glove_mean_pool(tokens, embeddings):
    vectors = [
        embeddings[token]
        for token in tokens
        if token in embeddings
    ]

    if not vectors:
        return np.zeros(embeddings.vector_size)

    return np.mean(vectors, axis=0)

def glove_max_pool(tokens, embeddings):
    vectors = [
        embeddings[token]
        for token in tokens
        if token in embeddings
    ]

    if not vectors:
        return np.zeros(embeddings.vector_size)

    return np.max(vectors, axis=0)

# Create a mean and max pooled vector for each document, for both embedding models
X_glove_mean = np.vstack([
    glove_mean_pool(tokens, glove)
    for tokens in df["glove_tokens"]
])

X_glove_max = np.vstack([
    glove_max_pool(tokens, glove)
    for tokens in df["glove_tokens"]
])

print(X_glove_mean.shape)
print(X_glove_max.shape)

(7516, 100)
(7516, 100)


In [66]:
# Glove coverage
glove_covered_tokens = sum(
    token in glove
    for tokens in df["glove_tokens"]
    for token in tokens
)

glove_total_tokens = sum(
    len(tokens)
    for tokens in df["glove_tokens"]
)

print(
    f"GloVe token coverage: "
    f"{glove_covered_tokens / glove_total_tokens:.2%}"
)

GloVe token coverage: 99.37%


In [67]:
# Training classifiers
y = df["emotion"]

implementations = {
    "GloVe mean": X_glove_mean,
    "GloVe max": X_glove_max,
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for name, X in implementations.items():
    classifier = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    scores = cross_val_score(
        classifier,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    print(
        f"{name}: "
        f"mean={scores.mean():.4f}, "
        f"std={scores.std():.4f}"
    )

GloVe mean: mean=0.4867, std=0.0158
GloVe max: mean=0.4207, std=0.0100


In [ ]:
import os
results_folder = "../Data/Results"
os.makedirs(results_folder, exist_ok=True)

In [ ]:
# Willen jullie dit testen a.u.b.?
import part2
import itertools

glove = api.load("glove-wiki-gigaword-100", return_path=True)
trained_emb_model = part2.clslib.PretrainedEmbeddings(glove, 100)
tokenizer_for_lstm = part2.clslib.BasicTokenizer(
    True, False, False, 'english', trained_emb_model)

def cross_validation_split(pairs, k, seed=0):
    """
    Shuffles a list of labelled texts and splits the result into k folds.
    
    pairs: lists of tuples (text, label)
    k: number of folds
    seed: for reproducible shuffling
    """
    
    # Shuffle the pairs
    random.seed(seed)
    pairs = list(pairs)
    ids = list(range(len(pairs)))
    random.shuffle(ids)

    # Split without coercing (text, int) tuples into a string array
    ids_parts = np.array_split(np.array(ids, dtype=object), k)
    
    # Create a dictionary to store the training and validation sets for each fold
    cross_validation_sets = {}
    for i in range(k):
        validation_set = [pairs[j] for j in ids_parts[i]] 
        training_ids = list(itertools.chain.from_iterable(part for k, part in enumerate(ids_parts) if k != i))
        training_set = [pairs[j] for j in training_ids]
        cross_validation_sets[i] = {
            "training": training_set, 
            "validation": validation_set, 
            'training_ids': training_ids,   # these are not often needed, but we return it just in case
            'validation_ids': ids_parts[i]  # these are not often needed, but we return it just in case
        }

    return cross_validation_sets

pairs=list(zip(df["content"], df["emotion"].map(list(emotions).index)))
splits = cross_validation_split(pairs, k=5)

part2.experiment.cross_validation(
        outcome_space=emotions,
        data_splits=splits,
        tokenizer=tokenizer_for_lstm,
        emb_model=trained_emb_model,
        enc_type='lstm', 
        enc_kwargs={'freeze_embeddings': True, 'hidden_size': 20, 'word_dropout': 0.},
        regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
        opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 20},    
        device='cuda', #'cpu'
        save_to=f"{results_folder}/lstm",
    )